In [ ]:
import pandas as pd
import numpy as np
import time
from datetime import datetime
import os
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import SGDClassifier

import shap

In [ ]:
df_adb_raw = pd.read_csv('final_adb_means.csv')
df_non_adb_raw = pd.read_csv('final_non_adb_means.csv')

df_adb = df_adb_raw.iloc[:, 5:]
df_adb['adb'] = 1
df_non_adb = df_non_adb_raw.iloc[:, 5:]
df_non_adb['adb'] = 0
df = pd.concat([df_adb, df_non_adb], ignore_index=True)
df = df.dropna()
input_feat = df.columns.tolist()[:-1]
print(len(df))

In [ ]:
print(df)
df.to_csv('presentation.csv', index=False)

In [ ]:
x = df.iloc[:, 0:26]  
y = df.iloc[:, 26] 

In [ ]:
train_r = 0.8
test_r = 0.2

smote = SMOTE(random_state=123)
x_balanced, y_balanced = smote.fit_resample(x, y)

x_train, x_test, y_train, y_test = train_test_split(x_balanced, y_balanced, test_size=1 - train_r, random_state=129)


print('train')
print('dataset:', len(x_train))
print('have:', sum(y_train))
print('No:', len(y_train) - sum(y_train))
print('------------')
print('test')
print('dataset:', len(x_test))
print('have:', sum(y_test))
print('No:', len(y_test) - sum(y_test))

In [ ]:
xgboost_grid = {"subsample":[0.5, 0.75, 1], 'max_features':['auto','sqrt','log2'],"max_depth":[2,6,10],"criterion":['friedman_mse','squared_error'] , "n_estimators":[250,500]}
KNN_grid = {'n_neighbors': list(range(0, len(input_feat)//2)), 'weights':['uniform', 'distance']}
NB_grid = {'var_smoothing': np.logspace(0,-9, num=100)}
RF_grid = {'criterion':['gini','entropy'], 'max_features':['auto','sqrt','log2'],'n_estimators':[250,500],"max_depth":[2,6,10]}
SVM_grid = {'C': [0.1, 1.0, 10],
            'gamma': ['scale', 'auto'],
            'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'max_iter': [1000, 5000, 10000],
            'tol': [1e-3, 1e-4, 1e-5]}
LR_grid ={'C': [0.1, 1.0, 10],'solver': ['lbfgs','liblinear'],'max_iter':[1000]}

In [ ]:
# prepare models
models = []

# models.append(('NB', GaussianNB(),NB_grid))
# models.append(('LR', LogisticRegression(), LR_grid))
# models.append(('RF', RandomForestClassifier(),RF_grid))
models.append(('Xgboost', GradientBoostingClassifier(), xgboost_grid))
# models.append(('KNN', KNeighborsClassifier(),KNN_grid))
# models.append(('SVM', SVC(kernel = 'rbf'), SVM_grid))

In [ ]:
results = []
names = []
scoring = 'accuracy'
for name, model, grid in models:
    grid = GridSearchCV(model, param_grid=grid, cv=10, verbose=0, n_jobs=-1, scoring=['precision', 'recall', 'accuracy', 'f1', 'roc_auc'], refit="accuracy")
    grid_search = grid.fit(x_train, y_train.ravel())
    print(name)
    print('best params:', grid.best_params_)
    print("Best score: %0.3f" % grid_search.best_score_)
    cv_results = pd.DataFrame(grid_search.cv_results_)
   
    best_model_results = cv_results.loc[grid_search.best_index_]
    print(round(best_model_results['mean_test_precision'] * 100, 2), '±', round(best_model_results['std_test_precision'] * 100, 2))
    print(round(best_model_results['mean_test_recall'] * 100, 2), '±', round(best_model_results['std_test_recall'] * 100, 2))
    print(round(best_model_results['mean_test_accuracy'] * 100, 2), '±', round(best_model_results['std_test_accuracy'] * 100, 2))
    print(round(best_model_results['mean_test_f1'] * 100, 2), '±', round(best_model_results['std_test_f1'] * 100, 2))
    print(round(best_model_results['mean_test_roc_auc'] * 100, 2), '±', round(best_model_results['std_test_roc_auc'] * 100, 2))
    print("")

In [ ]:
clf = GradientBoostingClassifier(criterion='friedman_mse', max_depth=10, max_features='auto', n_estimators=500, subsample=0.5)
# clf = RandomForestClassifier(criterion='gini', max_depth=10, max_features='auto', n_estimators=500)
clf_fit = clf.fit(x_train, y_train.ravel()) # training the data

test_y_predicted = clf_fit.predict(x_test)
test_y_predicted_proba = clf_fit.predict_proba(x_test)
y_predprob = (test_y_predicted_proba[:,1])

print(round(precision_score(y_test, test_y_predicted, average='macro') * 100, 2))
print(round(recall_score(y_test, test_y_predicted, average='macro') * 100, 2))
print(round(accuracy_score(y_test, test_y_predicted) * 100, 2))
print(round(f1_score(y_test, test_y_predicted, average='weighted') * 100, 2))
print(round(roc_auc_score(y_test, y_predprob, multi_class='ovr') * 100, 2))

In [ ]:
cm = confusion_matrix(y_test, test_y_predicted)

print("Confusion Matrix:")
print(cm)

In [ ]:
explainer = shap.TreeExplainer(clf_fit)

feature_names = x.columns.tolist()
print(feature_names)

In [ ]:
# Calculate SHAP values for the test data
shap_values = explainer.shap_values(x_test)
# Generate the SHAP summary plot as a density scatter plot
shap.summary_plot(shap_values, features=x_test, feature_names=feature_names, plot_type='dot', max_display=4)
plt.title("SHAP Summary Plot")
# Show the plot
plt.show()

In [ ]:
# param_grid = {
#     'loss': ['epsilon_insensitive'],
#     'penalty': ['l2', 'l1','elasticnet'],
#     'alpha': [0.0001],
#     'learning_rate': ['constant', 'adaptive'],
#     'eta0': [0.01, 0.1, 1.0],
#     'max_iter': [1000, 2000, 5000],
#     'tol': [1e-3, 1e-4, 1e-5]
# }

# models = []
# models.append(('SGD', SGDClassifier(loss='epsilon_insensitive'), param_grid))

# results = []
# names = []
# scoring = 'accuracy'

# for name, model, grid in models:
#     grid = GridSearchCV(model, param_grid = grid, cv=10, verbose=0, n_jobs=-1, scoring=['precision', 'recall', 'accuracy', 'f1', 'roc_auc'], refit="accuracy")
#     grid_search = grid.fit(x_train_balanced, y_train_balanced.ravel())
#     print(name)
#     print('best params:', grid.best_params_)
#     print("Best score: %0.3f" % grid_search.best_score_)
#     cv_results = pd.DataFrame(grid_search.cv_results_)

#     best_model_results = cv_results.loc[grid_search.best_index_]
#     print(round(best_model_results['mean_test_precision'] * 100, 2), '±', round(best_model_results['std_test_precision'] * 100, 2))
#     print(round(best_model_results['mean_test_recall'] * 100, 2), '±', round(best_model_results['std_test_recall'] * 100, 2))
#     print(round(best_model_results['mean_test_accuracy'] * 100, 2), '±', round(best_model_results['std_test_accuracy'] * 100, 2))
#     print(round(best_model_results['mean_test_f1'] * 100, 2), '±', round(best_model_results['std_test_f1'] * 100, 2))
#     print(round(best_model_results['mean_test_roc_auc'] * 100, 2), '±', round(best_model_results['std_test_roc_auc'] * 100, 2))
#     print("")

In [ ]:
from sklearn.preprocessing import StandardScaler
LR_grid ={'C': [0.1, 1.0, 10],'solver': ['lbfgs','liblinear'],'max_iter':[1000]}
df_adb_raw = pd.read_csv('final_adb_means.csv')
df_non_adb_raw = pd.read_csv('final_non_adb_means.csv')

df_adb = df_adb_raw.iloc[:, 5:]
df_adb['adb'] = 1
df_non_adb = df_non_adb_raw.iloc[:, 5:]
df_non_adb['adb'] = 0
df = pd.concat([df_adb, df_non_adb], ignore_index=True)
df = df.dropna()

input_feat = df.columns.tolist()[:-1]
print(input_feat)

x = df.iloc[:, 0:26]
y = df.iloc[:, 26]

train_r = 0.8
test_r = 0.2

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=1 - train_r, random_state=129)

# Scale the features
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

smote = SMOTE(random_state=123)
x_train_balanced, y_train_balanced = smote.fit_resample(x_train_scaled, y_train)

print('train')
print('dataset:', len(x_train_balanced))
print('have:', sum(y_train_balanced))
print('No:', len(y_train_balanced) - sum(y_train_balanced))
print('------------')
print('test')
print('dataset:', len(x_test))
print('have:', sum(y_test))
print('No:', len(y_test) - sum(y_test))

SVM_grid = {
    'C': [0.1, 1.0, 10],
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf'],
    'max_iter': [1000, 5000, 10000],
    'tol': [1e-3, 1e-4, 1e-5],
    'decision_function_shape': ['ovo', 'ovr'],
    'class_weight': [None, 'balanced'],
    'shrinking': [True, False]
}

# prepare models
models = []

models.append(('SVM', SVC(kernel='rbf'), SVM_grid))

results = []
names = []
scoring = 'accuracy'
for name, model, grid in models:
    grid = GridSearchCV(model, param_grid=grid, cv=10, verbose=0, n_jobs=-1,
                        scoring=['precision', 'recall', 'accuracy', 'f1', 'roc_auc'], refit="accuracy")
    grid_search = grid.fit(x_train_balanced, y_train_balanced.ravel())
    print(name)
    print('best params:', grid.best_params_)
    print("Best score: %0.3f" % grid_search.best_score_)
    cv_results = pd.DataFrame(grid_search.cv_results_)

    best_model_results = cv_results.loc[grid_search.best_index_]
    print(round(best_model_results['mean_test_precision'] * 100, 2), '±', round(best_model_results['std_test_precision'] * 100, 2))
    print(round(best_model_results['mean_test_recall'] * 100, 2), '±', round(best_model_results['std_test_recall'] * 100, 2))
    print(round(best_model_results['mean_test_accuracy'] * 100, 2), '±', round(best_model_results['std_test_accuracy'] * 100, 2))
    print(round(best_model_results['mean_test_f1'] * 100, 2), '±', round(best_model_results['std_test_f1'] * 100, 2))
    print(round(best_model_results['mean_test_roc_auc'] * 100, 2), '±', round(best_model_results['std_test_roc_auc'] * 100, 2))
    print("")


In [ ]:
SVM_grid = {'C': [0.1, 1.0, 10],
            'gamma': ['scale','auto'],
            'kernel': ['rbf'],
            'max_iter': [1000, 5000, 10000],
            'tol': [1e-3, 1e-4, 1e-5]}

# prepare models
models = []

models.append(('SVM', SVC(kernel = 'rbf'), SVM_grid))


results = []
names = []
scoring = 'accuracy'
for name, model, grid in models:
    grid = GridSearchCV(model, param_grid=grid, cv=10, verbose=0, n_jobs=-1, scoring=['precision', 'recall', 'accuracy', 'f1', 'roc_auc'], refit="accuracy")
    grid_search = grid.fit(x_train_balanced, y_train_balanced.ravel())
    print(name)
    print('best params:', grid.best_params_)
    print("Best score: %0.3f" % grid_search.best_score_)
    cv_results = pd.DataFrame(grid_search.cv_results_)
   
    best_model_results = cv_results.loc[grid_search.best_index_]
    print(round(best_model_results['mean_test_precision'] * 100, 2), '±', round(best_model_results['std_test_precision'] * 100, 2))
    print(round(best_model_results['mean_test_recall'] * 100, 2), '±', round(best_model_results['std_test_recall'] * 100, 2))
    print(round(best_model_results['mean_test_accuracy'] * 100, 2), '±', round(best_model_results['std_test_accuracy'] * 100, 2))
    print(round(best_model_results['mean_test_f1'] * 100, 2), '±', round(best_model_results['std_test_f1'] * 100, 2))
    print(round(best_model_results['mean_test_roc_auc'] * 100, 2), '±', round(best_model_results['std_test_roc_auc'] * 100, 2))
    print("")

In [ ]:
from sklearn.preprocessing import StandardScaler

df_adb_raw = pd.read_csv('final_adb_means.csv')
df_non_adb_raw = pd.read_csv('final_non_adb_means.csv')

df_adb = df_adb_raw.iloc[:, 5:]
df_adb['adb'] = 1
df_non_adb = df_non_adb_raw.iloc[:, 5:]
df_non_adb['adb'] = 0
df = pd.concat([df_adb, df_non_adb], ignore_index=True)
df = df.dropna()

input_feat = df.columns.tolist()[:-1]
print(input_feat)

x = df.iloc[:, 0:26]
y = df.iloc[:, 26]

train_r = 0.8
test_r = 0.2

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=1 - train_r, random_state=129)

# Scale the features
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

smote = SMOTE(random_state=123)
x_train_balanced, y_train_balanced = smote.fit_resample(x_train_scaled, y_train)

print('train')
print('dataset:', len(x_train_balanced))
print('have:', sum(y_train_balanced))
print('No:', len(y_train_balanced) - sum(y_train_balanced))
print('------------')
print('test')
print('dataset:', len(x_test))
print('have:', sum(y_test))
print('No:', len(y_test) - sum(y_test))

LR_grid = {
    'C': [0.1, 1.0, 10],
    'solver': ['lbfgs', 'liblinear'],
    'max_iter': [1000],
    'penalty': ['l1', 'l2'],  # Additional regularization type
    'fit_intercept': [True, False],  # Include or exclude intercept term
    'class_weight': [None, 'balanced']  # Weighting strategy for imbalanced classes
    # Add more hyperparameters as desired
}

# prepare models
models = []

models.append(('LR', LogisticRegression(), LR_grid))

results = []
names = []
scoring = 'accuracy'
for name, model, grid in models:
    grid = GridSearchCV(model, param_grid=grid, cv=10, verbose=0, n_jobs=-1,
                        scoring=['precision', 'recall', 'accuracy', 'f1', 'roc_auc'], refit="accuracy")
    grid_search = grid.fit(x_train_balanced, y_train_balanced.ravel())
    print(name)
    print('best params:', grid.best_params_)
    print("Best score: %0.3f" % grid_search.best_score_)
    cv_results = pd.DataFrame(grid_search.cv_results_)

    best_model_results = cv_results.loc[grid_search.best_index_]
    print(round(best_model_results['mean_test_precision'] * 100, 2), '±', round(best_model_results['std_test_precision'] * 100, 2))
    print(round(best_model_results['mean_test_recall'] * 100, 2), '±', round(best_model_results['std_test_recall'] * 100, 2))
    print(round(best_model_results['mean_test_accuracy'] * 100, 2), '±', round(best_model_results['std_test_accuracy'] * 100, 2))
    print(round(best_model_results['mean_test_f1'] * 100, 2), '±', round(best_model_results['std_test_f1'] * 100, 2))
    print(round(best_model_results['mean_test_roc_auc'] * 100, 2), '±', round(best_model_results['std_test_roc_auc'] * 100, 2))
    print("")